In [1]:
import numpy as np
import pandas as pd

In [2]:
FOLD = 7

In [3]:
flat_regions_path = f"/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/genomic_flat_regions/flat_regions_chrom_states_tsv/fold{FOLD}_selected_genomic_windows_centered_chrom_states.tsv"

In [4]:
df = pd.read_csv(flat_regions_path, sep="\t")

In [5]:
import torch

In [6]:
from typing import Optional

In [7]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
# def splice_and_to_string(main_path: str,
#                          slice_path: str,
#                          replace_bin: int = 320,
#                          bin_size: int = 2048,
#                          trim_bp: int = 131072,
#                          alphabet: str = "ACGT",
#                          device: Optional[str] = None):
#     """
#     Load a one-hot DNA tensor (shape: [4, L]), replace one 2048-bp bin in the middle,
#     trim `trim_bp` bases from both ends, and convert to an A/C/G/T string.

#     Parameters
#     ----------
#     main_path : str
#         Path to the main .pt tensor (e.g. 'chr11_65677312_66988032_X.pt').
#     slice_path : str
#         Path to the .pt tensor providing the replacement slice
#         (e.g. 'chr11_65677312_66988032_slice.pt').
#     replace_bin : int
#         Index of the 2048-bp bin to replace (default: 320).
#     bin_size : int
#         Size of each bin in bp (default: 2048).
#     trim_bp : int
#         Number of bp to cut from each side after replacement (default: 131072).
#     alphabet : str
#         Order of nucleotides corresponding to the one-hot channels (default: "ACGTN").
#         Change if your tensor channel order differs.

#     Returns
#     -------
#     seq_str : str
#         The trimmed sequence converted to letters.
#     """

#     main = torch.load(main_path, map_location=device)
#     slc  = torch.load(slice_path, map_location=device)

#     if isinstance(main, torch.Tensor):
#         main = main.detach().clone().requires_grad_(False)
#     if isinstance(slc, torch.Tensor):
#         slc = slc.detach().clone().requires_grad_(False)

#     if main.dim() != 3 or main.size(1) != len(alphabet):
#         raise ValueError(f"Expected main tensor of shape [4, L]. Got {tuple(main.shape)}")

#     start = replace_bin * bin_size
#     end   = (replace_bin + 1) * bin_size

#     if end > main.size(2):
#         raise ValueError(f"Replacement range [{start}:{end}] exceeds sequence length {main.size(2)}")
    
#     # Check slice sizes
#     if slc.dim() != 3 or slc.size(1) != len(alphabet) or slc.size(2) != (end - start):
#         raise ValueError(
#             f"Slice tensor must have shape [1, {len(alphabet)}, {end-start}], got {tuple(slc.shape)}"
#         )

#     # Replace
#     main[:, :, start:end] = slc

#     # Trim
#     L = main.size(2)
#     if 2 * trim_bp >= L:
#         raise ValueError(f"trim_bp ({trim_bp}) is too large for sequence length {L}")
#     trimmed = main[:, :, trim_bp : L - trim_bp]

#     # Convert to string
#     # Argmax over channels -> indices 0..3
#     idx = trimmed.argmax(dim=1).cpu().numpy()
    
#     letters = np.array(list(alphabet))
#     seq_arr = letters[idx]

#     if seq_arr.ndim > 1:
#         seq_arr = seq_arr.ravel()
    
#     seq_str = "".join(seq_arr.tolist())

#     return seq_str


In [8]:
import numpy as np
import re
from typing import Optional

def neutralize_homopolymers(seq_str: str, min_len: int = 3) -> str:
    """
    Identifies G or C homopolymers >= min_len and replaces them 
    with a 'GC' repeat of the same length.
    """
    def gc_replacer(match):
        hit = match.group(0)
        length = len(hit)
        # Create a GC repeat (e.g., GGG -> GCG, GGGG -> GCGC)
        replacement = ("GC" * (length // 2 + 1))[:length]
        return replacement

    # Regex looks for 3 or more G's OR 3 or more C's
    pattern = f"G{{{min_len},}}|C{{{min_len},}}"
    return re.sub(pattern, gc_replacer, seq_str)

def splice_and_to_string(main_path: str,
                         slice_path: str,
                         replace_bin: int = 320,
                         bin_size: int = 2048,
                         trim_bp: int = 131072,
                         alphabet: str = "ACGT",
                         device: Optional[str] = None,
                         fix_homopolymers: bool = True):
    """
    Loads one-hot DNA, replaces bin 320, trims, converts to string,
    and optionally replaces G/C homopolymers with GC repeats.
    """
    main = torch.load(main_path, map_location=device)
    slc  = torch.load(slice_path, map_location=device)

    # Detach and clone
    main = main.detach().clone().squeeze(0) if main.dim() == 3 else main.detach().clone()
    slc = slc.detach().clone().squeeze(0) if slc.dim() == 3 else slc.detach().clone()

    # Re-alignment logic (ensuring [4, L])
    if main.shape[0] != 4: main = main.permute(1, 0)
    if slc.shape[0] != 4: slc = slc.permute(1, 0)

    start = replace_bin * bin_size
    end   = (replace_bin + 1) * bin_size

    # 1. Perform the replacement in the center bin
    main[:, start:end] = slc

    # 2. Trim the flanking regions
    L = main.size(1)
    trimmed = main[:, trim_bp : L - trim_bp]

    # 3. Convert to string
    idx = trimmed.argmax(dim=0).cpu().numpy()
    letters = np.array(list(alphabet))
    seq_str = "".join(letters[idx].tolist())

    # 4. Neutralize G/C Homopolymers (the "GC-repeat" fix)
    if fix_homopolymers:
        seq_str = neutralize_homopolymers(seq_str, min_len=3)

    return seq_str

In [9]:
def save_to_fasta(seq: str, fasta_path: str, header: str = "sequence"):
    """
    Save a sequence string to a FASTA file.
    
    Parameters
    ----------
    seq : str
        The sequence (A/C/G/T/N).
    fasta_path : str
        Path to save the FASTA file.
    header : str
        The FASTA header (default: "sequence").
    """
    with open(fasta_path, "w") as f:
        f.write(f">{header}\n")
        # Write 80 characters per line (FASTA convention)
        for i in range(0, len(seq), 80):
            f.write(seq[i:i+80] + "\n")

In [10]:
for i, row in enumerate(df.itertuples(index=False)):
    
    print(f"seq {i}")
    
    chrom, start, end = row.chrom, row.centered_start, row.centered_end
    
    seq = splice_and_to_string(main_path = f"/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/suppressing_CTCFs/ohe_X/fold{FOLD}/{chrom}_{start}_{end}_X.pt",
                         slice_path = f"/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/suppressing_CTCFs/results_repeated/fold{FOLD}/{chrom}_{start}_{end}_slice.pt",
                         device = device)

    save_to_fasta(seq = seq, fasta_path=f"/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/alpha_genome_validation/ctct_suppressing/fold{FOLD}_homopolymers/{chrom}_{start}_{end}.fasta", header=f"{chrom}_{start}_{end}")

seq 0
seq 1
seq 2
seq 3
seq 4
seq 5
seq 6
seq 7
seq 8
seq 9
seq 10
seq 11
seq 12
seq 13
seq 14
seq 15
seq 16
seq 17
seq 18
seq 19
seq 20
seq 21
seq 22
seq 23
seq 24
seq 25
seq 26
seq 27
seq 28
seq 29
seq 30
seq 31
seq 32
seq 33
seq 34
seq 35
seq 36
seq 37
seq 38
seq 39
seq 40
